https://data.cityofnewyork.us/resource/erm2-nwe9.json

In [8]:
import requests
import pandas as pd

url = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

In [9]:
from ingest import ingest_data

df, metadata = ingest_data(url)

2026-11-24 14:10:25,995 | INFO | ingest | Starting ingestion from https://data.cityofnewyork.us/resource/erm2-nwe9.json (limit=1000)


2026-11-24 14:10:33,001 | INFO | ingest | Ingested 1000 rows and 49 columns


In [10]:
#!pip install great_expectations
#!pip install pyyaml

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [15]:
from validate import load_expectations,validate_dataframe

yaml = load_expectations()
result = validate_dataframe(df, yaml)

from pprint import pprint
pprint(result)

# for c in result["checks"]:
#     if c['status'] == "FAIL":
#         print(c)

2026-11-24 14:17:56,950 | INFO | great_expectations.data_context.types.base | Created temporary directory 'C:\Users\marwa\AppData\Local\Temp\tmpdwfxrz73' for ephemeral docs site


2026-11-24 14:17:56,952 | INFO | great_expectations.datasource.fluent.config | Loading 'datasources' ->
[]


Calculating Metrics:   0%|          | 0/98 [00:00<?, ?it/s]

{'checks': [{'actual': 1000,
             'check_name': 'expect_table_row_count_to_be_between',
             'column': '__dataset__',
             'details': 'unexpected_count=0',
             'expected': {'batch_id': 'pandas_source-dataframe_asset',
                          'min_value': 100},
             'status': 'PASS'},
            {'actual': 0.0,
             'check_name': 'expect_column_values_to_not_be_null',
             'column': 'unique_key',
             'details': 'unexpected_count=0',
             'expected': {'batch_id': 'pandas_source-dataframe_asset'},
             'status': 'PASS'},
            {'actual': 0.0,
             'check_name': 'expect_column_values_to_be_unique',
             'column': 'unique_key',
             'details': 'unexpected_count=0',
             'expected': {'batch_id': 'pandas_source-dataframe_asset'},
             'status': 'PASS'},
            {'actual': 0.0,
             'check_name': 'expect_column_values_to_not_be_null',
             'colu

# parent table schema:
- unique_identifier (pk) text
- row_count int
- timestamp timestamp
- dataset text
- falied_checks int
- overall_status text
- pass_rate decimal
- passed_checks int
- total_checks int 

# child table schema:
- id INTEGER PRIMARY KEY AUTOINCREMENT
- unique_identifier (fk) text
- expected decimal
- unexpected_count int
- column text
- check_name text
- actual decimal 
- status text

In [16]:
from history import save_run, get_recent_runs, analyze_trends

# 1. Persist the healthy baseline run
run_id_1 = save_run(result, metadata)
print("Saved Run 1:", run_id_1)

# 2. Check cold-start behavior (pass_rate_delta should be None)
analyze_trends(run_id_1)

Saved Run 1: 84d82c18-7e40-4933-9d2f-aa3c7e1c11d4


{'current_run_id': '84d82c18-7e40-4933-9d2f-aa3c7e1c11d4',
 'total_runs_recorded': 1,
 'current_pass_rate': 95.0,
 'previous_pass_rate': None,
 'pass_rate_delta': None,
 'avg_baseline_pass_rate': None,
 'avg_baseline_row_count': None,
 'row_count_pct_change': None,
 'worsening_checks': []}

In [17]:
from datetime import datetime, timezone
from validate import validate_dataframe

# 1. Corrupt a copy of df (inject 250 nulls into incident_zip + drop 200 rows)
bad_df = df.iloc[:800].copy()
bad_df.loc[:250, "incident_zip"] = None

bad_metadata = {
    **metadata,
    "ingested_at": datetime.now(timezone.utc).isoformat(),
    "row_count": len(bad_df),
}
bad_result = validate_dataframe(bad_df, yaml)

# 2. Save degraded run & inspect trends
run_id_2 = save_run(bad_result, bad_metadata)
print("Recent runs:", len(get_recent_runs()))
analyze_trends(run_id_2)

2026-11-26 09:15:14,830 | INFO | great_expectations.data_context.types.base | Created temporary directory 'C:\Users\marwa\AppData\Local\Temp\tmp6s0qr1ks' for ephemeral docs site
2026-11-26 09:15:14,859 | INFO | great_expectations.datasource.fluent.config | Loading 'datasources' ->
[]


Calculating Metrics:   0%|          | 0/98 [00:00<?, ?it/s]

Recent runs: 2


{'current_run_id': '97dd538b-bfea-4d36-952f-3f6150430068',
 'total_runs_recorded': 2,
 'current_pass_rate': 90.0,
 'previous_pass_rate': 95.0,
 'pass_rate_delta': -5.0,
 'avg_baseline_pass_rate': 95.0,
 'avg_baseline_row_count': 1000.0,
 'row_count_pct_change': -20.0,
 'worsening_checks': [{'column': 'closed_date',
   'check_name': 'expect_column_values_to_not_be_null',
   'previous': 73.2,
   'current': 76.625},
  {'column': 'incident_zip',
   'check_name': 'expect_column_values_to_not_be_null',
   'previous': 1.0999999999999999,
   'current': 32.125},
  {'column': 'latitude',
   'check_name': 'expect_column_values_to_not_be_null',
   'previous': 2.4,
   'current': 2.625},
  {'column': 'longitude',
   'check_name': 'expect_column_values_to_not_be_null',
   'previous': 2.4,
   'current': 2.625},
  {'column': 'x_coordinate_state_plane',
   'check_name': 'expect_column_values_to_not_be_null',
   'previous': 2.4,
   'current': 2.625},
  {'column': 'y_coordinate_state_plane',
   'check_nam

In [18]:
from report import generate_html_report

recent_runs = get_recent_runs(limit=10)
trend_summary = analyze_trends(run_id_2)

report_path = generate_html_report(bad_result, bad_metadata, trend_summary, recent_runs)
print("Generated report:", report_path)

Generated report: reports\quality_report_20261126_075407.html
